# Problem 17

- [ ] Trace through `coprime_reconstruct_frequency` for inputs $M=7$, $N=11$, $k_n=5$, and $k_m=3$ and verify your
    hand calculations from parts a) and b). Make sure to color-code output using the color scheme specified below:
   - $(M,N)$ - (`COLOR_PALETTE["electric_blue"]`)
   - $(k_m, k_n)$ - `COLOR_PALETTE["coral_orange"]`
   - $k_\text{true}$ - `COLOR_PALETTE["vivid_pink"]'`
   - $(a,b)$ - `COLOR_PALETTE["bright_green"]`

In [1]:
from __future__ import annotations

from kuramoto.cli.style import (
    PRESETS,
    cli_banner,
    cli_echo,
    cli_divider,
    cli_section,
    cli_gradient,
    COLOR_PALETTE,
    multi_gradient,
    style_coprime_coeff,
    style_coprime_gcd,
    style_coprime_remainder,
    style_coprime_term,
    style_crt_channel,
)
from kuramoto.solvers.coprime import coprime_reconstruct_frequency, extended_gcd


def trace_crt_reconstruction(
    k_m: int,
    k_n: int,
    M: int,
    N: int,
) -> int:
    """
    Trace the Chinese Remainder Theorem reconstruction step by step.

    Given coprime strides M and N and observed frequency aliases k_m (mod M)
    and k_n (mod N), this function prints a color-coded trace of the CRT
    formula and returns the reconstructed true frequency index.

    Parameters
    ----------
    k_m : int
        Observed alias modulo M (from Sampler B).
    k_n : int
        Observed alias modulo N (from Sampler A).
    M : int
        Subsampling stride for Sampler B (coprime to N).
    N : int
        Subsampling stride for Sampler A (coprime to M).

    Returns
    -------
    int
        The reconstructed true frequency index k_true in [0, M*N - 1].
    """
    modulus = M * N
    cli_section(f"CRT Reconstruction: ({style_coprime_term(M)}, {style_coprime_term(N)})")

    # Step 1: Bézout identity
    label_bezout = multi_gradient("Step 1 — Bézout Identity:", PRESETS["eigenscribe"])
    cli_echo(f"  {label_bezout}")
    g, a, b = extended_gcd(M, N)
    cli_echo(
        f"    ({style_coprime_coeff(a)}){style_coprime_term(M)} + ({style_coprime_coeff(b)}){style_coprime_term(N)} = {style_coprime_gcd(g)}"
    )
    cli_echo(
        f"    Bézout coefficients: {style_coprime_coeff('a')}={style_coprime_coeff(a)}, "
        f"{style_coprime_coeff('b')}={style_coprime_coeff(b)}"
    )
    cli_divider()

    # Step 2: Channel A contribution (k_n * a * M)
    label_a = multi_gradient("Step 2 — Channel A contribution:", PRESETS["tse-gradient-2"])
    cli_echo(f"  {label_a}")
    term_a = k_n * a * M
    cli_echo(
        f"    Term A = {style_coprime_remainder(k_n)} × ({style_coprime_coeff(a)}) × {style_coprime_term(M)} "
        f"= {style_crt_channel(term_a, 'A')}"
    )
    cli_echo(
        f"    (Recovers remainder mod {style_coprime_term(N)}: "
        f"{style_coprime_coeff(a)} × {style_coprime_term(M)} ≡ 1 (mod {style_coprime_term(N)}))"
    )
    cli_divider()

    # Step 3: Channel B contribution (k_m * b * N)
    label_b = multi_gradient("Step 3 — Channel B contribution:", PRESETS["tse-gradient-2"])
    cli_echo(f"  {label_b}")
    term_b = k_m * b * N
    cli_echo(
        f"    Term B = {style_coprime_remainder(k_m)} × ({style_coprime_coeff(b)}) × {style_coprime_term(N)} "
        f"= {style_crt_channel(term_b, 'B')}"
    )
    cli_echo(
        f"    (Recovers remainder mod {style_coprime_term(M)}: "
        f"{style_coprime_coeff(b)} × {style_coprime_term(N)} ≡ 1 (mod {style_coprime_term(M)}))"
    )
    cli_divider()

    # Step 4: Merge
    label_merge = multi_gradient("Step 4 — Merge:", PRESETS["tse-gradient-2"])
    cli_echo(f"  {label_merge}")
    k_true = (term_a + term_b) % modulus
    cli_echo(
        f"    k_true = ({style_crt_channel(term_a, 'A')} + {style_crt_channel(term_b, 'B')}) % {style_coprime_term(modulus)} "
        f"= {style_coprime_gcd(k_true)}"
    )
    cli_divider()

    # Step 5: Verification
    label_verify = multi_gradient("Step 5 — Verification:", PRESETS["tse-gradient-2"])
    cli_echo(f"  {label_verify}")
    check_m = k_true % M
    check_n = k_true % N
    match_m = "✔️" if check_m == k_m else "✖️"
    match_n = "✔️" if check_n == k_n else "✖️"
    cli_echo(
        f"    {style_coprime_gcd(k_true)} mod {style_coprime_term(M)} = {style_coprime_gcd(check_m)}  "
        f"{match_m} (matches k_m={style_coprime_remainder(k_m)})"
    )
    cli_echo(
        f"    {style_coprime_gcd(k_true)} mod {style_coprime_term(N)} = {style_coprime_gcd(check_n)}  "
        f"{match_n} (matches k_n={style_coprime_remainder(k_n)})"
    )
    cli_divider()

    # Cross-check with the library function
    k_lib = coprime_reconstruct_frequency(k_m=k_m, k_n=k_n, M=M, N=N)
    lib_match = "✔️" if k_lib == k_true else "✖️"
    cli_echo(
        f"  Library coprime_reconstruct_frequency() = {style_coprime_gcd(k_lib)}  {lib_match}"
    )

    return k_true


def trace_crt_reconstruction_error(
    k_m: int,
    k_n: int,
    M: int,
    N: int,
) -> None:
    """
    Trace what happens when M and N are not coprime.

    Demonstrates why coprimality is required for unique CRT reconstruction.
    """
    cli_section(f"CRT Reconstruction (Non-Coprime): ({style_coprime_term(M)}, {style_coprime_term(N)})")
    g, a, b = extended_gcd(M, N)
    cli_echo(
        f"  gcd({style_coprime_term(M)}, {style_coprime_term(N)}) = {style_coprime_gcd(g)}"
    )
    if g != 1:
        cli_echo(
            f"  ✖️ Strides are not coprime (gcd = {style_coprime_gcd(g)}). "
            f"CRT reconstruction is ambiguous."
        )
        cli_echo(
            f"  Multiple indices in [0, {M * N - 1}] share the same remainder pair "
            f"({style_coprime_remainder(k_m)}, {style_coprime_remainder(k_n)})."
        )
        # Show the ambiguous indices
        ambiguous = [t for t in range(M * N) if t % M == k_m and t % N == k_n]
        if ambiguous:
            cli_echo(f"  Matching indices: {ambiguous}")
        else:
            cli_echo(f"  No index matches — the remainder pair is inconsistent.")
    else:
        cli_echo(f"  ✔️ Strides are coprime. Proceeding with reconstruction.")
    cli_divider()

    # Attempt reconstruction — should raise ValueError
    try:
        k_true = coprime_reconstruct_frequency(k_m=k_m, k_n=k_n, M=M, N=N)
        cli_echo(f"  Result: {style_coprime_gcd(k_true)}")
    except ValueError as exc:
        cli_echo(f"  ValueError: {exc}")

KeyError: 'orange_yellow'

In [ ]:
# Problem 17 example: k_n=5 (mod 11), k_m=3 (mod 7)
k_true = trace_crt_reconstruction(k_m=3, k_n=5, M=7, N=11)
print(f"\nReconstructed k_true = {k_true}")

In [ ]:
# Non-coprime counterexample
trace_crt_reconstruction_error(k_m=0, k_n=0, M=6, N=9)

In [29]:
# Another coprime example
k_true_2 = trace_crt_reconstruction(k_m=2, k_n=7, M=5, N=8)
print(f"\nReconstructed k_true = {k_true_2}")


 ──────────────────────────────────────────────────
  CRT Reconstruction: (5, 8)
 ──────────────────────────────────────────────────
  Step 1 — Bézout Identity:
    (-3)5 + (2)8 = 1
    Bézout coefficients: a=-3, b=2
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 2 — Channel A contribution:
    Term A = 7 × (-3) × 5 = -105
    (Recovers remainder mod 8: -3 × 5 ≡ 1 (mod 8))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 3 — Channel B contribution:
    Term B = 2 × (2) × 8 = 32
    (Recovers remainder mod 5: 2 × 8 ≡ 1 (mod 5))
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 4 — Merge:
    k_true = (-105 + 32) % 40 = 7
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Step 5 — Verification:
    7 mod 5 = 2  ✔️ (matches k_m=2)
    7 mod 8 = 7  ✔️ (matches k_n=7)
 ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─ ─
  Library coprime_reconstruct_frequency() = 7  ✔️

Reconstructed k_true = 7


## Part 2
- [ ] Use `coprime_snapshot_steps(m, n)` in `crt_stride_visualization.py` to print out channel A strides,
  channel B strides, and their union in a gradient of colors corresponding with the channel A color and channel
  B color as defined in @src/kuramoto/cli/style/domain.py.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Ensure the sibling module is importable
_here = Path(__file__).parent if "__file__" in globals() else Path.cwd()
_mod_dir = _here / "SECTION_C" / "CODING_TASKS"
if str(_mod_dir) not in sys.path:
    sys.path.insert(0, str(_mod_dir))

from crt_stride_visualization import coprime_snapshot_steps

from kuramoto.cli.style import (
    cli_divider,
    cli_echo,
    cli_section,
    multi_gradient,
    style_crt_channel,
)
from kuramoto.cli.style.domain import _CRT_CHANNEL_A, _CRT_CHANNEL_B, _ansi_truecolor
from eigenscribe.frontend.colors import COLOR_PALETTE, hex_to_rgb


# Channel colors from domain.py
CHAN_A_HEX: str = f"#{_CRT_CHANNEL_A[0]:02x}{_CRT_CHANNEL_A[1]:02x}{_CRT_CHANNEL_A[2]:02x}"
CHAN_B_HEX: str = f"#{_CRT_CHANNEL_B[0]:02x}{_CRT_CHANNEL_B[1]:02x}{_CRT_CHANNEL_B[2]:02x}"

# Gradient: Channel A color -> Channel B color
_AB_GRADIENT = (CHAN_A_HEX, CHAN_B_HEX)

# Header colors
IBM_WHITE_RGB = hex_to_rgb(COLOR_PALETTE["ibm_white"])
SKY_BLUE_RGB = hex_to_rgb(COLOR_PALETTE["eigen_blue"])
STUDIO_PURPLE_RGB = hex_to_rgb(COLOR_PALETTE["studio_purple"])
BRIGHT_GREEN_RGB = hex_to_rgb(COLOR_PALETTE["bright_green"])


def _build_header(m: int, n: int, total: int) -> str:
    """Build the section header with IBM-white text and colored values.

    Parameters
    ----------
    m : int
        Channel A stride.
    n : int
        Channel B stride.
    total : int
        Coprime period (m * n).

    Returns
    -------
    str
        ANSI-escaped header string.
    """
    white = _ansi_truecolor(IBM_WHITE_RGB, "Coprime Snapshot Strides  (")
    m_colored = _ansi_truecolor(SKY_BLUE_RGB, f"m={m}")
    comma1 = _ansi_truecolor(IBM_WHITE_RGB, ", ")
    n_colored = _ansi_truecolor(STUDIO_PURPLE_RGB, f"n={n}")
    comma2 = _ansi_truecolor(IBM_WHITE_RGB, ", ")
    period_label = _ansi_truecolor(BRIGHT_GREEN_RGB, "period")
    eq = _ansi_truecolor(IBM_WHITE_RGB, "=")
    period_value = _ansi_truecolor(BRIGHT_GREEN_RGB, str(total))
    close = _ansi_truecolor(IBM_WHITE_RGB, ")")
    return f"{white}{m_colored}{comma1}{n_colored}{comma2}{period_label}{eq}{period_value}{close}"


def print_coprime_strides(m: int = 7, n: int = 11) -> None:
    """Print coprime-stride snapshot steps with gradient-colored labels.

    Channel A strides are colored in slime_green, Channel B strides in
    golden_amber, and the union steps in a gradient between the two.

    Parameters
    ----------
    m : int
        Channel A stride (default 7).
    n : int
        Channel B stride (default 11).
    """
    chan_a, chan_b, union_steps = coprime_snapshot_steps(m, n)
    total = m * n

    cli_section(_build_header(m, n, total))

    # --- Channel A strides ---
    label_a = style_crt_channel(
        f"Channel A (every {m}-th step) — {len(chan_a)} snapshots:",
        "A",
    )
    cli_echo(f"  {label_a}")
    cli_echo(f"    {chan_a}")
    cli_divider()

    # --- Channel B strides ---
    label_b = style_crt_channel(
        f"Channel B (every {n}-th step) — {len(chan_b)} snapshots:",
        "B",
    )
    cli_echo(f"  {label_b}")
    cli_echo(f"    {chan_b}")
    cli_divider()

    # --- Union (gradient of A -> B) ---
    label_union = multi_gradient(
        f"Union (A ∪ B) — {len(union_steps)} snapshots (saved {total + 1 - len(union_steps)}):",
        _AB_GRADIENT,
    )
    cli_echo(f"  {label_union}")
    cli_echo(f"    {union_steps}")
    cli_divider()

    # --- Overlap highlight ---
    overlap = sorted(set(chan_a) & set(chan_b))
    cli_echo(
        f"  Overlap (both channels fire): {multi_gradient(str(overlap), _AB_GRADIENT)}"
    )


print_coprime_strides(m=7, n=11)

## Part 3: CRT coprime-stride visualization

Diagnostic-first visual sanity checks for coprime-stride snapshot wrapper (the memory-saving $\mathcal{O}(m+n)$ recorder used in this repo).

In [ ]:
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# ------------------------------------------------------------------------------
# Palette: prefer the project's colors.py (golden-easing engine), else fallback.
# ------------------------------------------------------------------------------

try:
    import colors as cs

    def hex_to_rgb01(hex_color: str) -> tuple[float, float, float]:
        r, g, b = cs.hex_to_rgb(hex_color)
        return (r / 255.0, g / 255.0, b / 255.0)

    def golden_gradient(stops: list[str], n: int) -> list[tuple[float, float, float]]:
        """Golden-eased gradient through hex stops, normalized to [0, 1]^3."""
        rgb255 = cs.generate_golden_ratio_gradient(stops, length=n)
        return [(r / 255.0, g / 255.0, b / 255.0) for r, g, b in rgb255]

    USING_PROJECT_COLORS = True
except ImportError:  # graceful fallback: same palette, plain linear interp
    USING_PROJECT_COLORS = False

    def hex_to_rgb01(hex_color: str) -> tuple[float, float, float]:
        h = hex_color.lstrip("#")
        return tuple(int(h[i : i + 2], 16) / 255.0 for i in (0, 2, 4))

    def golden_gradient(stops: list[str], n: int) -> list[tuple[float, float, float]]:
        rgbs = [hex_to_rgb01(s) for s in stops]
        segs = len(rgbs) - 1
        out = []
        for i in range(n):
            p = i / (n - 1) * segs
            k = min(int(p), segs - 1)
            f = p - k
            out.append(
                tuple(rgbs[k][c] + (rgbs[k + 1][c] - rgbs[k][c]) * f for c in range(3))
            )
        return out

# Semantic colors (sam names as colors.py)
CALM = COLOR_PALETTE["cyan_green"]      # healthy / low residual
WARN = COLOR_PALETTE["warning_amber"]   # rising residual
ALERT = COLOR_PALETTE["error_crimson"]  # drift detected
INK = COLOR_PALETTE["ibm_black"]        # background / text

### 2.A Is my sampling correct? - CRT hit-map
- [ ] Create a $7 \times 11$ grid where cell $(a,b)$ is colored by the unique time step $t\in[0, 77)$ satisfying $t=a\mod{7}$ and $t=b\mod{11}$.
    - The Chinese Remainder Theorem (CRT) guarantees this mapping is a bijection, so coloring by $t$ should show a perfectly regular,
    non-repeating permutation pattern. Any visual irregularity = broken stride logic.

In [ ]:
# ---------------------------------------------------------------------------
# CRT machinery — the core of the whole visualization
# ---------------------------------------------------------------------------

M, N = 7, 11        # coprime strides
TOTAL = M * N       # coprime period

def crt_cell_to_step(a: int, b: int) -> int:
    """Unique t in [0, M*N) with t = a (mod M) and t = b (mod N).

    Standard CRT cconstruction:
        t = a * N * inv(N mod M) + b * M * inv(M mod N)    (mod M*N)
    Valid precisely because gcd(M, N) == 1.
    """
    inv_n_mod_m = pow(N, -1, M)     # modular inverse of N mod M
    inv_m_mod_n = pow(M, -1, N)     # modular inverse of M mod N
    return (a * N * inv_n_mod_m + b * M * inv_m_mod_n) % TOTAL

def build_step_grid() -> np.ndarray:
    """(M x N) array; entry (a, b) = the time step that CRT cell represents."""
    return np.array(
        [[crt_cell_to_step(a, b) for b in range(N)] for a in range(M)], dtype=int
    )

def coprime_snapshot_steps() -> tuple[list[int], list[int], list[int]]:
    """Steps recorded by each channel, plus their union (sorted).

    Channel A records every M-th step, channel B every N-th step.
    Memory: len(union) snapshots instead of TOTAL + 1 dense ones.
    """
    chan_a = list(range(0, TOTAL + 1, M))    # includes t = TOTAL
    chan_b = list(range(0, TOTAL + 1, N))    # includes t = TOTAL
    union = sorted(set(chan_a) | set(chan_b))
    return chan_a, chan_b, union

In [ ]:
# ---------------------------------------------------------------------------
# Toy physics: a Kuramoto-style rotor, healthy vs. frequency-drifting
# ---------------------------------------------------------------------------

OMEGA = 2.0 * np.pi / 17.0    # base angular velocity (deliberately non-aligned with M or N)

def simulate(drift: float) -> np.ndarray:
    """True phase theta_t at every dense step t in [0, TOTAL].

    drift > 0 adds a quadratic phase term theta += 0.5 * drift * t^2, i.e. the instantaneous frequency creeps
    upward -- stand-in for bivector-torque accumulation error in a geometric-algebra rotor.
    """
    t = np.arange(TOTAL + 1, dtype=float)
    return OMEGA * t + 0.5 * drift * t * t

def reconstruct_from_snapshots(theta_true: np.ndarray, union: list[int]) -> np.ndarray:
    """Rebuild the full phase curve from coprime-stride samples only.

    We pretend we only know theta at the union steps (as teh recorder does), interpolate linearly between them,
    and return the reconstruction on all steps. The truth array is used ONLY to score the residual afterwards.
    """
    t_stored = np.array(union, dtype=float)
    theta_stored = theta_true[union]
    t_dense = np.arange(TOTAL + 1, dtype=float)
    return np.interp(t_dense, t_stored, theta_stored)

In [ ]:
# ---------------------------------------------------------------------------
# Color encodings
# ---------------------------------------------------------------------------

def residual_colormap() -> LinearSegmentedColormap:
    """cyan_green -> warning_ember -> error_crimson (golden-eased via colors.py)."""
    grad = golden_gradient([CALM, WARN, ALERT], 256)
    return LinearSegmentedColormap.from_list("crt_residual", grad)

def phase_to_hue01(theta: float) -> float:
    """Wrap a phase to [0, 1) so hue is a *cyclic* phase encoding."""
    return (theta / (2.0 * np.pi)) % 1.0

# ---------------------------------------------------------------------------
# Panels
# ---------------------------------------------------------------------------

def draw_panel1(ax, step_grid: np.ndarray) -> None:
    """Panel 1: coloir each CRT cell by the time step it represents."""
    grad = np.array(golden_gradient(
        [
            COLOR_PALETTE["midnight_navy"], COLOR_PALETTE["eigen_blue"],
            COLOR_PALETTE["cosmic_iris"], COLOR_PALETTE["vivid_pink"]
         ], 256
    ))
    norm = step_grid / (TOTAL - 1)
    img = grad[(norm * 255).astype(int)]
    ax.imshow(img, interpolation="nearest")
    ax.set_title(
        "Panel 1 - CRT hit-map\n"
        f"{M}x{N} grid, cell color = unique step t (bijection => regular pattern)",
        fontsize=10,
    )
    ax.set_xlabel("residue mod N (=11) -> channel B stride")
    ax.set_ylabel("residue mod M (=7) -> channel A stride")
    # Annotate a few cells so the CRT correspondence is visible
    for a in range(0, M, 2):
        for b in range(0, N, 3):
            t = step_grid[a, b]
            ax.text(b, a, str(t), ha="center", va="center", fontsize=7, color="white")

def draw_panel2(fig_axes, step_grid: np.ndarray, drift_values: list[float]) -> None:
    """Panel 2: reconstruction residual per CRT cell, healthy vs drifted."""
    cmap = residual_colormap()
    vmax = 0.0
    res_grids = []
    for drift in drift_values:
        theta = simulate(drift)
        _, _, union = coprime_snapshot_steps()
        recon = reconstruct_from_snapshots(theta, union)
        resid = np.abs(recon - theta)
        vmax = max(vmax, resid.max())
        # scatter residuals onto the CRT grid
        grid = np.zeros((M, N))
        for a in range(M):
            for b in range(N):
                grid[a, b] = resid[step_grid[a, b]]
        res_grids.append(grid)
    for ax, grid, drift in zip(fig_axes, res_grids, drift_values):
        im = ax.imshow(grid, cmap=cmap, interpolation="nearest", vmin=0, vmax=vmax)
        ax.set_title(
            f"Panel 2 - residual (drift={drift})\n"
            "cool = reconstruction faithful, warm = drift",
            fontsize=9,
        )
        ax.set_xticks([])
        ax.set_yticks([])
    return im  # for the shared colorbar

def draw_panel3(fig_axes, step_grid: np.ndarray, drift_values: list[float]) -> None:
    """Panel 3: cyclic hue encoding of the *reconstructed* phase field."""
    for ax, drift in zip(fig_axes, drift_values):
        theta = simulate(drift)
        _, _, union = coprime_snapshot_steps()
        recon = reconstruct_from_snapshots(theta, union)
        grid = np.zeros((M, N, 3))
        import colorsys

        for a in range(M):
            for b in range(N):
                hue = phase_to_hue01(recon[step_grid[a, b]])
                r, g, bcol = colorsys.hsv_to_rgb(hue, 0.75, 0.9)
                grid[a, b] = (r, g, bcol)
        ax.imshow(grid, interpolation="nearest")
        ax.set_title(
            f"Panel 3 - phase hue (drift={drift}\n"
            "smooth winding = healthy; hue shear = frequency drift",
            fontsize=9,
        )
        ax.set_xticks([])
        ax.set_yticks([])

In [ ]:
# ---------------------------------------------------------------------------
# Driver
# ---------------------------------------------------------------------------

def main() -> None:
    step_grid = build_step_grid()
    chan_a, chan_b, union = coprime_snapshot_steps()

    # -- sanity checks you want printed before trusting any plot ---
    assert len(set(step_grid.flatten())) == TOTAL, "CRT bijection broken!"
    saved = (TOTAL + 1) - len(union)
    print(f"Dense snapshots would be: {TOTAL + 1}")
    print(f"Channel A ({M}-stride): {len(chan_a)} | Channel B ({N}-stride): {len(chan_b)}")
    print(f"Coprime union stored: {len(union)} (saved {saved}, "
          f"~{saved / (TOTAL + 1):.0%} reduction)")
    print(f"Using project colors.py: {USING_PROJECT_COLORS}")

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.patch.set_facecolor(INK)

    draw_panel1(axes[0, 0], step_grid)
    im = draw_panel2([axes[0, 1], axes[0, 2]], step_grid, drift_values=[0.0, 0.002])
    fig.colorbar(im, ax=axes[0, 2], fraction=0.046)
    draw_panel3([axes[1, 1], axes[1, 2]], step_grid, drift_values=[0.0, 0.002])

    # Bottom-left: explain the memory saving in one glance
    ax = axes[1, 0]
    ax.axis("off")
    ax.text(
        0.02, 0.85,
        "Memory bookkeeping",
        fontsize=12, color="white", weight="bold",
    )
    ax.text(
        0.02, 0.55,
        f"dense:    O(mn)  = {TOTAL + 1} snapshots\n"
        f"coprime:  O(m + n) = {len(union)} snapshots\n\n"
        f"channel A fires at t = 0 (mod {M})\n"
        f"channel B fires at t= 0 (mod {N})\n"
        f"overlap only at t = 0 (mod {TOTAL})",
        fontsize=10, color=COLOR_PALETTE["ice_blue"], family="monospace",
    )

    fig.suptitle(
        "Coprime-stride CRT diagnostics for the Kuramoto rotor benchmark",
        color="white", fontsize=13, y=0.98,
    )
    plt.tight_layout()
    # plt.savefig("crt_stride_diagnostics.png", dpi=150, facecolor=INK)
    plt.show()

main()